# 08 — Model Evaluation and SHAP Explainability

This notebook provides comprehensive evaluation of the trained models:
1. Confusion matrices
2. ROC curves
3. Precision-Recall curves
4. SHAP summary plot (global feature importance)
5. SHAP waterfall plot for a sample patient

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from sklearn.base import clone
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, auc,
    precision_recall_curve, average_precision_score,
)

from src.config import DATASETS, SEED
from src.data.loader import load_raw_dataset, get_target_column
from src.data.preprocessor import MediSensePreprocessor
from src.data.splitter import stratified_split
from src.models.base_learners import get_base_learners
from src.evaluation.metrics import compute_metrics, format_metrics

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

## 1. Prepare Data (Heart Disease)

In [ ]:
dataset_name = 'heart'
df = load_raw_dataset(dataset_name)
target_col = get_target_column(dataset_name)

# Clean
df['ca'] = pd.to_numeric(df['ca'], errors='coerce')
df['thal'] = pd.to_numeric(df['thal'], errors='coerce')
df['target'] = (df['target'] > 0).astype(int)

# Split
train_df, val_df, test_df = stratified_split(df, target_col)

X_train = train_df.drop(columns=[target_col])
y_train = train_df[target_col].values
X_test = test_df.drop(columns=[target_col])
y_test = test_df[target_col].values

# Preprocess
preprocessor = MediSensePreprocessor(dataset_name)
X_train_proc = preprocessor.fit_transform(X_train, y_train)
X_test_proc = preprocessor.transform(X_test)
feature_names = preprocessor.get_feature_names_out()

print(f"Train: {X_train_proc.shape}, Test: {X_test_proc.shape}")
print(f"Features: {feature_names}")

## 2. Train All Models

In [ ]:
trained_models = {}
predictions = {}

for name, learner in get_base_learners():
    fitted = clone(learner)
    fitted.fit(X_train_proc, y_train)
    trained_models[name] = fitted
    
    y_pred = fitted.predict(X_test_proc)
    y_prob = fitted.predict_proba(X_test_proc)[:, 1] if hasattr(fitted, 'predict_proba') else None
    predictions[name] = {'y_pred': y_pred, 'y_prob': y_prob}
    
    metrics = compute_metrics(y_test, y_pred, y_prob)
    print(f"{name.upper()}:")
    print(format_metrics(metrics))
    print()

## 3. Confusion Matrices

In [ ]:
model_names = list(trained_models.keys())
n_models = len(model_names)

fig, axes = plt.subplots(1, n_models, figsize=(4 * n_models, 4))

for i, name in enumerate(model_names):
    ax = axes[i]
    cm = confusion_matrix(y_test, predictions[name]['y_pred'])
    disp = ConfusionMatrixDisplay(cm, display_labels=['No Disease', 'Disease'])
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(name.upper())

plt.suptitle('Confusion Matrices — Heart Disease (Test Set)', fontsize=14)
plt.tight_layout()
plt.show()

## 4. ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))

for name in model_names:
    y_prob = predictions[name]['y_prob']
    if y_prob is not None:
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        roc_auc_val = auc(fpr, tpr)
        ax.plot(fpr, tpr, label=f'{name.upper()} (AUC={roc_auc_val:.3f})', linewidth=2)

ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — Heart Disease (Test Set)')
ax.legend(loc='lower right')
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.05])
plt.tight_layout()
plt.show()

## 5. Precision-Recall Curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))

for name in model_names:
    y_prob = predictions[name]['y_prob']
    if y_prob is not None:
        precision, recall, _ = precision_recall_curve(y_test, y_prob)
        ap = average_precision_score(y_test, y_prob)
        ax.plot(recall, precision, label=f'{name.upper()} (AP={ap:.3f})', linewidth=2)

# Baseline: proportion of positive class
baseline = y_test.mean()
ax.axhline(y=baseline, color='k', linestyle='--', alpha=0.5, label=f'Baseline ({baseline:.2f})')

ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curves — Heart Disease (Test Set)')
ax.legend(loc='lower left')
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.05])
plt.tight_layout()
plt.show()

## 6. SHAP Analysis — Global Feature Importance

We use the XGBoost model for SHAP analysis since tree-based explainers are fast and well-supported.

In [ ]:
# Use XGBoost model for SHAP
xgb_model = trained_models['xgb']

# Create a DataFrame with feature names for better plots
X_test_df = pd.DataFrame(X_test_proc, columns=feature_names)

# Create SHAP explainer
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test_df)

print(f"SHAP values shape: {shap_values.shape}")
print(f"Base value (expected value): {explainer.expected_value:.4f}")

In [ ]:
# SHAP Summary Plot (Beeswarm)
fig, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(shap_values, X_test_df, show=False)
plt.title('SHAP Summary Plot — XGBoost (Heart Disease)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# SHAP Bar Plot (Mean absolute SHAP values)
fig, ax = plt.subplots(figsize=(10, 6))
shap.summary_plot(shap_values, X_test_df, plot_type='bar', show=False)
plt.title('Mean |SHAP Value| — Feature Importance (Heart Disease)', fontsize=14)
plt.tight_layout()
plt.show()

## 7. SHAP Waterfall Plot — Sample Patient Explanation

In [ ]:
# Pick a sample patient from the test set
sample_idx = 0
sample_pred = predictions['xgb']['y_pred'][sample_idx]
sample_prob = predictions['xgb']['y_prob'][sample_idx]
sample_true = y_test[sample_idx]

print(f"Sample Patient (index {sample_idx}):")
print(f"  True label: {sample_true} ({'Disease' if sample_true == 1 else 'No Disease'})")
print(f"  Predicted:  {sample_pred} (probability: {sample_prob:.4f})")
print(f"  Features:")
for fname, fval in zip(feature_names, X_test_proc[sample_idx]):
    print(f"    {fname}: {fval:.4f}")

In [ ]:
# SHAP Waterfall for the sample patient
shap_explanation = shap.Explanation(
    values=shap_values[sample_idx],
    base_values=explainer.expected_value,
    data=X_test_proc[sample_idx],
    feature_names=feature_names,
)

fig, ax = plt.subplots(figsize=(10, 8))
shap.plots.waterfall(shap_explanation, show=False)
plt.title(f'SHAP Waterfall — Sample Patient (True={sample_true}, Pred={sample_pred})', fontsize=13)
plt.tight_layout()
plt.show()

## 8. SHAP Analysis on Random Forest (for comparison)

In [ ]:
rf_model = trained_models['rf']
rf_explainer = shap.TreeExplainer(rf_model)
rf_shap_values = rf_explainer.shap_values(X_test_df)

# For binary classification, RF returns a list [class_0, class_1]
if isinstance(rf_shap_values, list):
    rf_shap_values = rf_shap_values[1]  # Take positive class

fig, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(rf_shap_values, X_test_df, show=False)
plt.title('SHAP Summary Plot — Random Forest (Heart Disease)', fontsize=14)
plt.tight_layout()
plt.show()

## Summary

**Model Performance:**
- All base learners achieve reasonable performance on the heart disease test set.
- ROC and PR curves provide a more nuanced view than single-point metrics.

**SHAP Insights:**
- The SHAP summary plot reveals which features drive predictions globally.
- The waterfall plot explains individual predictions, showing how each feature pushes the model output up or down.
- XGBoost and Random Forest generally agree on the most important features, increasing confidence in the explanations.